# 02 — Data preparation evidence

Preparation and validation evidence for the canonical geometry, governed CLC derivatives, and national panel. National preparation logic lives in reusable `src/` modules and scripts; this notebook does not modify raw or processed data.

In [ ]:
from pathlib import Path
import json
import sys
import pyogrio
from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CLC, SPATIAL, TEMPORAL
from src.feature_contract import FIELD_CONTRACTS, PREDICTOR_COLUMNS, TARGET_COLUMN
from src.geospatial_utils import GRID_PATH
from src.source_registry import CLC_PREPARED_PORTUGAL_LAYERS
print(SPATIAL)

## Feature-table contract

This is the analytical data contract used by the reusable pipeline. It makes explicit that the table has seven canonical predictors and one continuous T+1 target; geometry remains in the canonical grid GeoPackage rather than being repeated in every cell-year row.

In [ ]:
feature_contract = pd.DataFrame([
    {'column': name, 'role': 'target' if name == TARGET_COLUMN else 'predictor', 'unit': FIELD_CONTRACTS[name].unit, 'allowed range': f'{FIELD_CONTRACTS[name].minimum} to {FIELD_CONTRACTS[name].maximum}', 'source-year rule': FIELD_CONTRACTS[name].source_year_rule}
    for name in (*PREDICTOR_COLUMNS, TARGET_COLUMN)
])
display(feature_contract)
assert len(PREDICTOR_COLUMNS) == 7
print('Unique analytical key: cell_id × observation_year; target:', TARGET_COLUMN)

## Canonical grid geometry

In [ ]:
grid_info = pyogrio.read_info(GRID_PATH, layer='canonical_mainland_grid_1km')
assert grid_info['features'] == 89_112
assert grid_info['crs'] == 'EPSG:3763'
print({'path': GRID_PATH.relative_to(PROJECT_ROOT).as_posix(), 'layer': 'canonical_mainland_grid_1km', 'features': grid_info['features'], 'crs': grid_info['crs']})

## Governed Portugal CLC layers

In [ ]:
for year, record in CLC_PREPARED_PORTUGAL_LAYERS.items():
    path = PROJECT_ROOT / record.prepared_path
    facts = record.validation_facts
    info = pyogrio.read_info(path, layer=facts.layer_name)
    assert info['features'] == facts.feature_count
    assert info['crs'] == record.crs
    assert facts.class_code_field in info['fields']
    print(year, record.prepared_path, info['features'], info['crs'], 'registered validation: passed')

## CLC assignment and no-future-information check

CLC is retrospective broad landscape context. The assigned reference year must never be later than predictor year T; this table exposes the governed assignment rather than assuming annual land-cover change.

In [ ]:
clc_assignment = pd.DataFrame([
    {'predictor year T': year, 'CLC reference year': CLC.reference_year(year), 'prepared path': CLC.prepared_dataset(year)[0], 'layer': CLC.prepared_dataset(year)[1]}
    for year in range(TEMPORAL.predictor_start_year, TEMPORAL.predictor_end_year + 1)
])
assert (clc_assignment['CLC reference year'] <= clc_assignment['predictor year T']).all()
display(clc_assignment)

## National-panel validation evidence

In [ ]:
metrics_path = PROJECT_ROOT / 'data/processed/national_panel_2015_2024_validation.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
panel_summary = pd.DataFrame([
    {'check': 'canonical grid cells', 'value': metrics['grid_cell_count']},
    {'check': 'expected cell-year rows', 'value': metrics['expected_row_count']},
    {'check': 'actual cell-year rows', 'value': metrics['actual_row_count']},
    {'check': 'duplicate analytical keys', 'value': metrics['duplicate_analytical_key_count']},
    {'check': 'panel SHA-256', 'value': metrics['panel_sha256']},
])
assert metrics['expected_row_count'] == metrics['actual_row_count']
assert metrics['duplicate_analytical_key_count'] == 0
display(panel_summary)